<a href="https://colab.research.google.com/github/naman39910/voyage-analytics/blob/main/nootbooks%20/recommendation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**VOYAGE ANALYTICS - TRAVEL RECOMMENDATION MODEL**
## Goal : Recommend hotels to users based on their travel history

In [ ]:
import pandas as pd

## Load Datasets

First, let's load the `users.csv`, `flights.csv`, and `hotels.csv` files into pandas DataFrames.

In [ ]:
# Load users data
users_df = pd.read_csv('/content/users.csv')
display(users_df.head())

In [ ]:
# Load flights data
flights_df = pd.read_csv('/content/flights.csv')
display(flights_df.head())

In [ ]:
# Load hotels data
hotels_df = pd.read_csv('/content/hotels.csv')
display(hotels_df.head())

## Merging Datasets for Recommendation Model

To build a recommendation model, we typically need to create a unified dataset that links users to their historical interactions (flights, hotels). Here's a general strategy for merging these datasets:

1.  **Identify Common Keys:** Look for columns that exist in multiple DataFrames and can be used to link records (e.g., `user_id`, `hotel_id`, `flight_id`).

2.  **User-Flight-Hotel Link:** The most common approach is to link `users_df` with `flights_df` and `hotels_df` based on `user_id`. If `flights_df` and `hotels_df` contain `user_id` and also specific identifiers for the services (`flight_id`, `hotel_id`), we can perform merges.

    *   **Merge `users_df` with `flights_df`:** Use `user_id` as the key. This will tell us which users took which flights.
    *   **Merge the result with `hotels_df`:** If `flights_df` contains a `hotel_id` or a `destination_city` that can be linked to `hotels_df`, we can use that to connect the user's travel to hotels. Alternatively, if `hotels_df` has `user_id` for past bookings, merge directly.

    The goal is to get a table where each row represents a user's interaction with a hotel, potentially through a flight.

3.  **Consider Recommendation Approach:**
    *   **Content-Based:** If we want to recommend hotels similar to what a user has previously liked, we need user-hotel interaction data and hotel features.
    *   **Collaborative Filtering:** If we want to recommend hotels based on what similar users liked, we need a user-item (hotel) interaction matrix.

Let's assume `flights_df` contains `user_id` and potentially `destination_city` or `hotel_id`, and `hotels_df` contains `hotel_id` and `city` information. We can merge `users_df` with `flights_df` and then join the result with `hotels_df` on relevant keys.

To proceed, we should examine the columns in each DataFrame and determine the best merging strategy. We are looking for columns that act as foreign keys.

In [ ]:
merged_flights_hotels_df = pd.merge(flights_df, hotels_df, on=['travelCode', 'userCode'], how='inner')
display(merged_flights_hotels_df.head())

In [ ]:
final_merged_df = pd.merge(merged_flights_hotels_df, users_df, left_on='userCode', right_on='code', how='inner')
display(final_merged_df.head())

## Hotel Data Analysis

Let's begin by examining the merged dataset, `final_merged_df`, to understand its structure and content.

In [ ]:
# Display information about the DataFrame (column names, data types, non-null values)
print(final_merged_df.info())

In [ ]:
# Display descriptive statistics for numerical columns
display(final_merged_df.describe())

### Distribution of Hotels and Places

In [ ]:
# Count the occurrences of each hotel name (name_x from hotels_df)
hotel_name_counts = final_merged_df['name_x'].value_counts()
print('Top 10 Hotel Names:')
display(hotel_name_counts.head(10))

In [ ]:
# Count the occurrences of each place (hotel location)
hotel_place_counts = final_merged_df['place'].value_counts()
print('\nTop 10 Hotel Places:')
display(hotel_place_counts.head(10))

### Hotel Pricing and Stay Duration Analysis

In [ ]:
# Analyze daily hotel price (price_y)
print('Daily Hotel Price (price_y) Statistics:')
display(final_merged_df['price_y'].describe())

In [ ]:
# Analyze total hotel cost (total)
print('\nTotal Hotel Cost (total) Statistics:')
display(final_merged_df['total'].describe())

In [ ]:
# Analyze duration of stay (days)
print('\nHotel Stay Duration (days) Statistics:')
display(final_merged_df['days'].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.histplot(final_merged_df['price_y'], kde=True)
plt.title('Distribution of Daily Hotel Price')
plt.xlabel('Daily Price')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
sns.histplot(final_merged_df['total'], kde=True)
plt.title('Distribution of Total Hotel Cost')
plt.xlabel('Total Cost')
plt.ylabel('Frequency')

plt.subplot(1, 3, 3)
sns.histplot(final_merged_df['days'], bins=range(1, final_merged_df['days'].max() + 2), kde=False)
plt.title('Distribution of Hotel Stay Duration')
plt.xlabel('Days')
plt.ylabel('Frequency')
plt.xticks(range(1, final_merged_df['days'].max() + 1))

plt.tight_layout()
plt.show()

## Feature Engineering

To prepare the data for a recommendation model, we'll perform some feature engineering. This involves creating new features from existing ones to enhance the model's ability to learn patterns.

First, let's rename some columns to make them more descriptive and avoid confusion, especially after merging. We have `price_x` (flight price) and `price_y` (hotel daily price), `name_x` (hotel name) and `name_y` (user name), and `date_x` (flight date) and `date_y` (hotel booking date).

In [ ]:
final_merged_df.rename(columns={
    'price_x': 'flight_price',
    'time': 'flight_time',
    'distance': 'flight_distance',
    'name_x': 'hotel_name',
    'price_y': 'hotel_daily_price',
    'total': 'hotel_total_cost',
    'date_x': 'flight_date',
    'date_y': 'hotel_booking_date',
    'name_y': 'user_name'
}, inplace=True)

print("Columns renamed for clarity.")
display(final_merged_df.head())

Next, let's extract more granular date-related features like the day of the week and month from both `flight_date` and `hotel_booking_date`. These can capture weekly or monthly patterns in travel and hotel bookings.

In [ ]:
# Convert date columns to datetime objects
final_merged_df['flight_date'] = pd.to_datetime(final_merged_df['flight_date'])
final_merged_df['hotel_booking_date'] = pd.to_datetime(final_merged_df['hotel_booking_date'])

# Extract day of week and month for flight dates
final_merged_df['flight_day_of_week'] = final_merged_df['flight_date'].dt.dayofweek
final_merged_df['flight_month'] = final_merged_df['flight_date'].dt.month

# Extract day of week and month for hotel booking dates
final_merged_df['hotel_booking_day_of_week'] = final_merged_df['hotel_booking_date'].dt.dayofweek
final_merged_df['hotel_booking_month'] = final_merged_df['hotel_booking_date'].dt.month

print("Date-related features extracted.")
display(final_merged_df[['flight_date', 'flight_day_of_week', 'flight_month', 'hotel_booking_date', 'hotel_booking_day_of_week', 'hotel_booking_month']].head())

Finally, let's create a feature for the total flight cost, which can be useful for understanding the overall travel expenditure.

In [ ]:
# Assuming 'flight_price' is a round trip price, if not, we would need more info on flight segments
# For now, let's consider flight_price as the total flight cost.
final_merged_df['total_flight_cost'] = final_merged_df['flight_price']

print("Total flight cost feature added.")
display(final_merged_df[['flight_price', 'total_flight_cost']].head())

With these new features, our dataset is now better prepared for building a recommendation model. Let's review the updated `final_merged_df` information to see the changes and ensure everything is as expected.

In [ ]:
print(final_merged_df.info())
display(final_merged_df.head())

With these new features, our dataset is now better prepared for building a recommendation model. Let's review the updated `final_merged_df` information to see the changes and ensure everything is as expected.

In [ ]:
print(final_merged_df.info())
display(final_merged_df.head())

## Categorical Feature Encoding

Many machine learning algorithms cannot directly work with categorical data. Therefore, we need to convert these text-based categories into numerical representations. We will primarily use One-Hot Encoding for most nominal categorical features.

In [ ]:
# Identify categorical columns to be encoded
categorical_cols = [
    'from', 'to', 'flightType', 'agency', 'hotel_name', 'place',
    'company', 'user_name', 'gender'
]

# Apply One-Hot Encoding
final_merged_df_encoded = pd.get_dummies(final_merged_df, columns=categorical_cols, drop_first=True)

print(f"Original DataFrame shape: {final_merged_df.shape}")
print(f"Encoded DataFrame shape: {final_merged_df_encoded.shape}")

display(final_merged_df_encoded.head())

Now that we've encoded the categorical features, the dataset has expanded significantly, with each unique category in the specified columns becoming a new binary column. This `final_merged_df_encoded` DataFrame is now ready for further steps, such as defining the recommendation strategy or model training.

## Data Splitting for Model Evaluation

To evaluate the performance of our future recommendation model, we need to split our dataset into training and testing sets. The training set will be used to build the model, and the testing set will be used to assess how well the model generalizes to new, unseen data. We'll perform a standard random split.

In [ ]:
from sklearn.model_selection import train_test_split

# Define our features (X) and for now, we'll consider all columns in the encoded dataframe as features
# In a recommendation context, X might often be used to predict an interaction or a rating.
# For simplicity, we'll split the entire DataFrame for now, as the specific target for recommendation
# might be implicit (e.g., user-item interaction).
X = final_merged_df_encoded

# Split the data into training and testing sets (e.g., 80% train, 20% test)
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

print(f"Shape of training set: {X_train.shape}")
print(f"Shape of testing set: {X_test.shape}")

display(X_train.head())

We now have our `X_train` and `X_test` DataFrames, ready for the next steps in building and evaluating our recommendation model. The next steps will depend on the specific type of recommendation system we choose to implement (e.g., collaborative filtering, content-based, or hybrid).

## K-Nearest Neighbors (KNN) Recommendation System

To implement the KNN recommendation system, we first need to prepare our feature set by removing identifier columns and original date columns. Then, we'll scale the numerical features, and finally, train the `NearestNeighbors` model.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# Identify non-feature columns that should not be used in distance calculation
# 'code' is redundant with 'userCode' for identifying users
non_feature_cols = [
    'travelCode', 'userCode', 'code', 'flight_date', 'hotel_booking_date'
]

# Create feature-only DataFrames for training and testing
# We drop the non-feature columns from X_train and X_test
X_train_features = X_train.drop(columns=non_feature_cols, errors='ignore')
X_test_features = X_test.drop(columns=non_feature_cols, errors='ignore')

print(f"Shape of X_train_features: {X_train_features.shape}")
print(f"Shape of X_test_features: {X_test_features.shape}")

display(X_train_features.head())

### Feature Scaling

Now, we'll apply `StandardScaler` to our feature sets. Scaling is crucial for distance-based algorithms like KNN, as it ensures that features with larger values do not disproportionately influence the distance calculations.

In [ ]:
scaler = StandardScaler()

# Fit the scaler on the training features and transform both training and testing features
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled = scaler.transform(X_test_features)

# Convert scaled arrays back to DataFrames for easier handling
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_features.columns, index=X_train_features.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_features.columns, index=X_test_features.index)

print("Features scaled successfully.")
display(X_train_scaled_df.head())

### Train K-Nearest Neighbors Model

Next, we'll initialize and fit the `NearestNeighbors` model. This model will learn the underlying structure of our data to find the `k` most similar data points (travel packages) to any given query package.

In [ ]:
n_neighbors = 5 # Number of nearest neighbors to find
knn_model = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine') # Cosine similarity is good for high-dimensional data

# Fit the model on the scaled training data
knn_model.fit(X_train_scaled_df)

print(f"KNN model trained with {n_neighbors} neighbors using cosine similarity.")

### Generate Recommendations

Now, let's pick a random travel package from our test set and find its `k` nearest neighbors (i.e., recommended similar packages) from the training set. We'll display details of the query package and its recommendations.

In [ ]:
# Select a random travel package from the test set as our query
query_index = np.random.choice(X_test_scaled_df.index)
query_package_scaled = X_test_scaled_df.loc[[query_index]]

# Retrieve the original details of the query package from the UNENCODED DataFrame
query_package_original = final_merged_df.loc[[query_index]]

# Find the k nearest neighbors
distances, indices = knn_model.kneighbors(query_package_scaled)

# Retrieve the original details of the recommended packages from the UNENCODED DataFrame
# The indices returned by kneighbors are positions within the fitted data (X_train_scaled_df),
# so we need to map them back to the original DataFrame indices.
recommended_packages_original_indices = X_train_scaled_df.iloc[indices[0]].index
recommended_packages_original = final_merged_df.loc[recommended_packages_original_indices]


print("--- Query Travel Package ---")
display(query_package_original[['hotel_name', 'place', 'flightType', 'flight_price', 'hotel_total_cost', 'user_name', 'gender', 'age']])

print(f"\n--- Top {n_neighbors} Recommended Similar Travel Packages ---")
display(recommended_packages_original[['hotel_name', 'place', 'flightType', 'flight_price', 'hotel_total_cost', 'user_name', 'gender', 'age']])

## Recommendation System Evaluation

To evaluate our recommendation system, we can examine the similarity scores (distances) between the query package and the recommended packages. Lower distances indicate higher similarity. We'll also make the recommendation process a reusable function.

In [ ]:
def get_recommendations(query_original_index, query_package_scaled, model, X_train_scaled_df, original_data, n_recommendations=5):
    """
    Generates recommendations for a given query package.

    Args:
        query_original_index (int): The original index of the query package in the original_data.
        query_package_scaled (pd.DataFrame): The scaled feature DataFrame (single row) of the query package.
        model (NearestNeighbors): The trained KNN model.
        X_train_scaled_df (pd.DataFrame): The scaled feature DataFrame used for model training.
        original_data (pd.DataFrame): The original, unencoded DataFrame to retrieve full details.
        n_recommendations (int): The number of recommendations to return.

    Returns:
        tuple: A tuple containing the original query package (DataFrame),
               the original recommended packages (DataFrame), and their distances.
    """
    # Retrieve the original details of the query package
    query_package_original = original_data.loc[[query_original_index]]

    # Find the k nearest neighbors
    # The model was fitted on X_train_scaled_df, so it will search within that data
    distances, indices = model.kneighbors(query_package_scaled)

    # The indices returned by kneighbors are positions within the fitted data (X_train_scaled_df),
    # so we need to map them back to the original DataFrame indices.
    # We also exclude the query item itself if it's present in the recommendations (distance == 0)
    # The query package itself might be in X_train if the split was not completely disjoint for indices
    # or if we were querying the training set. For safety, we filter for distance > 1e-6.
    recommended_indices_in_scaled_data = indices[0][distances[0] > 1e-6]

    # Get the actual original indices from the X_train_scaled_df DataFrame
    recommended_packages_original_indices = X_train_scaled_df.iloc[recommended_indices_in_scaled_data].index

    # Retrieve the original details of the recommended packages
    recommended_packages_original = original_data.loc[recommended_packages_original_indices].head(n_recommendations)
    recommended_distances = distances[0][distances[0] > 1e-6][:n_recommendations]

    return query_package_original, recommended_packages_original, recommended_distances

print("get_recommendations function redefined.")

In [ ]:
# Select a random travel package from the test set as our query
query_index = np.random.choice(X_test_scaled_df.index)

# Get the scaled features of the query package from the test set
query_package_scaled = X_test_scaled_df.loc[[query_index]]

# Get recommendations using the new function
query_package_original, recommended_packages_original, recommended_distances = get_recommendations(
    query_index, query_package_scaled, knn_model, X_train_scaled_df, final_merged_df, n_recommendations=n_neighbors
)

print("--- Query Travel Package ---")
display(query_package_original[['hotel_name', 'place', 'flightType', 'flight_price', 'hotel_total_cost', 'user_name', 'gender', 'age']])

print(f"\n--- Top {n_neighbors} Recommended Similar Travel Packages (with Cosine Distances) ---")
# Add distances to the display for evaluation
recommended_packages_display = recommended_packages_original[['hotel_name', 'place', 'flightType', 'flight_price', 'hotel_total_cost', 'user_name', 'gender', 'age']].copy()
recommended_packages_display['Cosine_Distance'] = recommended_distances
display(recommended_packages_display)